#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("schemaBronze", "bronze")
dbutils.widgets.text("schemaSilver", "silver")
dbutils.widgets.text("catalogo", "project_dev")

In [0]:
schemaBronze = dbutils.widgets.get("schemaBronze")
schemaSilver = dbutils.widgets.get("schemaSilver")
catalogo = dbutils.widgets.get("catalogo")

# definir rutas
path_inicio = f"{catalogo}.{schemaBronze}"
path_target = f"{catalogo}.{schemaSilver}"

In [0]:
from pyspark.sql.functions import col, broadcast, when, lit
from pyspark.sql import functions as F

##products

In [0]:

df_products = spark.table( f"{catalogo}.{schemaBronze}.products ") \
  .select( "product_id", "product_category_name", "product_description_length")


##product category name translation

In [0]:
df_c_name_trlt = (
    spark.table( f" {catalogo}.{schemaBronze}.products_category")
    .select( 
            col("product_category_name").alias("product_category_llave"), 
            "product_category_name_english"
    )
)


## join

In [0]:
## hacer un join en las dos tablas
df_products = (
    df_products.alias("p").join( broadcast( df_c_name_trlt.alias("c")) , col("p.product_category_name") == col("c.product_category_llave"),how="left")
    .drop("product_category_llave", "product_description_length")
)


In [0]:
## crear las columnas condicionales
df_products = ( df_products
    .select( 
            "product_id",
            # col2
            when( col( "product_category_name").isNull(), lit( "unidentified"))
            .otherwise( col("product_category_name"))
            .alias("product_category_name"),
            #COL3
            when( (col( "product_category_name_english").isNull() & col("product_category_name").isNotNull()) , col( "product_category_name")) 
            .when(col("product_category_name_english").isNull(), lit( "unidentified"))
            .otherwise( col("product_category_name_english"))
            .alias("product_category_name_english")            
    )
)

In [0]:
### guardar la tabla clean en el target
df_products.write.mode("overwrite").insertInto(f"{path_target}.products_transform")

In [0]:
%sql
---- aplicar optimizer a la tabla target
OPTIMIZE ${catalogo}.${schemaSilver}.products_transform
ZORDER BY product_id

In [0]:
%sql
--- habilitar vacum dinamico
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
---- solo tener archivos inferiores  24 horas
VACUUM ${catalogo}.${schemaSilver}.products_transform RETAIN 24 HOURS

In [0]:
%sql
--- describir la historia de la tabla
DESCRIBE HISTORY ${catalogo}.${schemaSilver}.products_transform